In [1]:
# Practical: Implementation of Quantum Teleportation Algorithm

# Step 1: Import required libraries
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

# Step 2: Create a quantum circuit with 3 qubits and 3 classical bits
qc = QuantumCircuit(3, 3)

# Step 3: Prepare the unknown quantum state to be teleported
# Apply gates to create a quantum state on qubit 0
qc.ry(1.0, 0)
qc.rz(0.5, 0)
qc.barrier()

# Step 4: Create entanglement between Alice and Bob (Bell pair on qubits 1 and 2)
qc.h(1)
qc.cx(1, 2)
qc.barrier()

# Step 5: Alice performs Bell-state measurement
qc.cx(0, 1)
qc.h(0)
qc.barrier()

# Measure Alice's qubits into classical bits 0 and 1
qc.measure(0, 0)
qc.measure(1, 1)
qc.barrier()

# Step 6: Bob applies correction operations conditionally
with qc.if_test((qc.clbits[1], 1)):
    qc.x(2)

with qc.if_test((qc.clbits[0], 1)):
    qc.z(2)

qc.barrier()

# Step 7: Measure Bob's final qubit into classical bit 2
qc.measure(2, 2)

# Step 8: Display the quantum circuit diagram
print("Quantum Teleportation Circuit:")
print(qc.draw())

# Step 9: Execute the circuit using Aer Simulator
simulator = AerSimulator()
compiled_circuit = transpile(qc, simulator)

result = simulator.run(
    compiled_circuit,
    shots=1024
).result()

# Step 10: Display measurement results
counts = result.get_counts()
print("\nMeasurement Results:")
print(counts)

# Step 11: Display histogram
plot_histogram(counts)
plt.show()

Quantum Teleportation Circuit:
     ┌───────┐┌─────────┐ ░            ░      ┌───┐ ░ ┌─┐    ░                 »
q_0: ┤ Ry(1) ├┤ Rz(0.5) ├─░────────────░───■──┤ H ├─░─┤M├────░─────────────────»
     └───────┘└─────────┘ ░ ┌───┐      ░ ┌─┴─┐└───┘ ░ └╥┘┌─┐ ░                 »
q_1: ─────────────────────░─┤ H ├──■───░─┤ X ├──────░──╫─┤M├─░─────────────────»
                          ░ └───┘┌─┴─┐ ░ └───┘      ░  ║ └╥┘ ░   ┌──────  ┌───┐»
q_2: ─────────────────────░──────┤ X ├─░────────────░──╫──╫──░───┤ If-0  ─┤ X ├»
                          ░      └───┘ ░            ░  ║  ║  ░   └──╥───  └───┘»
                                                       ║  ║    ┌────╨────┐     »
c: 3/══════════════════════════════════════════════════╩══╩════╡ c_1=0x1 ╞═════»
                                                       0  1    └─────────┘     »
«                                          ░    
«q_0: ─────────────────────────────────────░────
«                                          ░    
«q_1: ──────